# Terra Signal: poder discriminatório das variáveis preditoras (Grupo 1)

Estudo individual de cada variável preditora contra o alvo (`Churn`), usando o **Information Value (IV)**. A interação entre variáveis preditoras fica para o notebook 05.

Cada pergunta tem três células: **pergunta** → **código** → **resposta**.

- Dados: `history.csv`, na raiz do repositório. O `inference.csv` não tem `Churn` e não entra aqui.
- A limpeza é a mesma do notebook 03.
- `CustomerFeedback` fica de fora: é texto livre e vaza o alvo (pergunta 30 do notebook 03).
- Roda com pandas e numpy, sem Spark. Funciona localmente e no Databricks (Git folder).

## Setup

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)

DATA_DIR = "../data/raw/"  # este notebook fica em notebooks/; os CSVs ficam em data/raw/
hist_raw = pd.read_csv(DATA_DIR + "history.csv")
print("history:", hist_raw.shape)

history: (5634, 23)


**Limpeza e variáveis.** A função `prep` é a do notebook 03: padroniza `PhoneService`, recupera `tenure` e imputa `TotalCharges`.

In [2]:
def prep(d):
    """Limpeza: padroniza PhoneService, recupera tenure e imputa TotalCharges."""
    d = d.copy()
    d["PhoneService"] = d["PhoneService"].str.capitalize()
    d["tenure_num"] = pd.to_numeric(d["tenure"], errors="coerce")
    d["tenure_num"] = d["tenure_num"].fillna((d["TotalCharges"] / d["MonthlyCharges"]).round())
    d["TotalCharges_imp"] = d["TotalCharges"].fillna(d["MonthlyCharges"] * d["tenure_num"])
    d.loc[d["tenure_num"] == 0, "TotalCharges_imp"] = 0
    return d


hist = prep(hist_raw)
y = (hist["Churn"] == "Yes").astype(int)

ADDONS = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
PREDITORAS = ["gender", "SeniorCitizen", "Partner", "Dependents", "tenure_num", "PhoneService", "MultipleLines",
              "InternetService"] + ADDONS + ["Contract", "PaperlessBilling", "PaymentMethod", "MonthlyCharges",
                                             "TotalCharges_imp", "MonthlyIncome"]
print("Variáveis preditoras:", len(PREDITORAS))
print(f"Clientes: {len(y)} | churners: {y.sum()} ({y.mean():.1%})")

Variáveis preditoras: 20
Clientes: 5634 | churners: 1504 (26.7%)


## Como o IV é calculado

Cada variável é dividida em faixas. As numéricas viram 10 faixas por quantil, e as categóricas usam as próprias categorias. Para cada faixa:

- **WOE** = ln(% dos não churners na faixa / % dos churners na faixa). Negativo indica risco acima da média.
- **Contribuição ao IV** = (% dos não churners − % dos churners) × WOE.

O IV da variável é a soma das contribuições das faixas. Interpretação usual (Siddiqi):

| IV | Poder preditivo |
|---|---|
| < 0,02 | Sem poder |
| 0,02 a 0,10 | Fraco |
| 0,10 a 0,30 | Médio |
| 0,30 a 0,50 | Forte |
| ≥ 0,50 | Muito forte |

In [3]:
N_FAIXAS = 10


def criar_faixas(coluna):
    """Decis para variáveis numéricas (com faixa própria para ausentes); categorias como estão nas demais."""
    if not pd.api.types.is_numeric_dtype(coluna) or coluna.nunique() <= N_FAIXAS:
        return coluna.astype(str)
    faixas = pd.qcut(coluna, N_FAIXAS, duplicates="drop")
    if faixas.isna().any():
        faixas = faixas.cat.add_categories("Ausente").fillna("Ausente")
    return faixas


def tabela_iv(coluna, alvo):
    """Devolve clientes, taxa de churn, WOE e contribuição ao IV de cada faixa da variável."""
    t = alvo.groupby(criar_faixas(coluna), observed=True).agg(clientes="size", churners="sum")
    t["nao_churners"] = t["clientes"] - t["churners"]
    t["taxa_churn"] = t["churners"] / t["clientes"]
    # Faixa sem churners ou sem não churners deixaria o WOE infinito; conta como 0,5 cliente.
    pct_churners = t["churners"].replace(0, 0.5) / t["churners"].sum()
    pct_nao_churners = t["nao_churners"].replace(0, 0.5) / t["nao_churners"].sum()
    t["woe"] = np.log(pct_nao_churners / pct_churners)
    t["iv"] = (pct_nao_churners - pct_churners) * t["woe"]
    return t


def poder_preditivo(iv):
    if iv < 0.02:
        return "sem poder"
    if iv < 0.10:
        return "fraco"
    if iv < 0.30:
        return "médio"
    if iv < 0.50:
        return "forte"
    return "muito forte"

### 1. Qual é o IV de cada variável preditora?

In [4]:
tabelas = {c: tabela_iv(hist[c], y) for c in PREDITORAS}
ranking = pd.DataFrame(
    [{"variavel": c, "faixas": len(t), "iv": t["iv"].sum()} for c, t in tabelas.items()]
).sort_values("iv", ascending=False, ignore_index=True)
ranking["poder"] = ranking["iv"].map(poder_preditivo)
ranking.round(4)

,variavel,faixas,iv,poder
0,Contract,3,1.1669,muito forte
1,tenure_num,11,0.7689,muito forte
2,OnlineSecurity,3,0.7233,muito forte
3,TechSupport,3,0.7132,muito forte
4,InternetService,3,0.6275,muito forte
5,OnlineBackup,3,0.5319,muito forte
6,DeviceProtection,3,0.5016,muito forte
7,PaymentMethod,4,0.4370,forte
8,StreamingMovies,3,0.3999,forte
9,StreamingTV,3,0.3954,forte


**Resposta:** `Contract` lidera com folga (IV 1,17), seguido de `tenure` (0,77).

- **Muito forte:** `Contract`, `tenure`, `OnlineSecurity` (0,72), `TechSupport` (0,71), `InternetService` (0,63), `OnlineBackup` (0,53) e `DeviceProtection` (0,50).
- **Forte:** `PaymentMethod` (0,44), `StreamingMovies` (0,40), `StreamingTV` (0,40), `MonthlyCharges` (0,38) e `TotalCharges` (0,31).
- **Médio:** `PaperlessBilling` (0,20), `Dependents` (0,16), `SeniorCitizen` (0,11) e `Partner` (0,11).
- **Sem poder:** `MultipleLines` (0,010), `MonthlyIncome` (0,004), `gender` (0,001) e `PhoneService` (0,001).

Nenhuma variável caiu na faixa fraca.

### 2. Como o churn se distribui nas faixas de cada variável?

In [5]:
for c in ranking["variavel"]:
    print(f"=== {c} | IV = {tabelas[c]['iv'].sum():.4f}")
    print(tabelas[c].round(4).to_string(), "\n")

=== Contract | IV = 1.1669
                clientes  churners  nao_churners  taxa_churn     woe      iv
Contract                                                                    
Month-to-month      3099      1322          1777      0.4266 -0.7144  0.3206
One year            1188       138          1050      0.1162  1.0191  0.1656
Two year            1347        44          1303      0.0327  2.3781  0.6807 

=== tenure_num | IV = 0.7689
               clientes  churners  nao_churners  taxa_churn     woe      iv
tenure_num                                                                 
(-0.001, 2.0]       674       392           282      0.5816 -1.3395  0.2577
(2.0, 6.0]          505       225           280      0.4455 -0.7915  0.0647
(6.0, 12.0]         577       204           373      0.3536 -0.4067  0.0184
(12.0, 20.0]        543       166           377      0.3057 -0.1899  0.0036
(20.0, 29.0]        551       131           420      0.2377  0.1549  0.0023
(29.0, 40.0]        582  

**Resposta:** os maiores riscos estão no contrato mensal (42,7%), nos 2 primeiros meses de `tenure` (58,2%) e no cheque eletrônico (45,1%).

- **`tenure`** cai de forma contínua, de 58,2% para 4,2% na faixa mais alta.
- **`MonthlyCharges` não é monotônica:** o churn sobe até 41,4% na faixa de R$ 94–103 e cai para 25,9% na faixa mais alta (R$ 103–119). Uma regressão logística não captaria isso com a variável contínua.
- **Um cliente** ficou sem `tenure` e sem `TotalCharges` depois da limpeza. Ele aparece na faixa `Ausente`, com impacto desprezível no IV (0,0009).

### 3. Que IV uma variável sem nenhum sinal teria com 10 faixas?

In [6]:
rng = np.random.default_rng(0)
ruido = [tabela_iv(pd.Series(rng.normal(size=len(y))), y)["iv"].sum() for _ in range(200)]
print(f"IV de ruído (200 simulações): média = {np.mean(ruido):.4f} | percentil 95 = {np.quantile(ruido, 0.95):.4f}")
print(f"MonthlyIncome: {tabelas['MonthlyIncome']['iv'].sum():.4f}")

IV de ruído (200 simulações): média = 0.0082 | percentil 95 = 0.0148
MonthlyIncome: 0.0043


**Resposta:** cerca de 0,008 em média, e não 0, porque dividir em faixas sempre captura um pouco de acaso. `MonthlyIncome` (0,004) fica abaixo dessa média: **não se distingue de ruído**, como o notebook 03 já indicava (pergunta 12).

### 4. O IV dos serviços adicionais está inflado pela categoria `No internet service`?

In [7]:
linhas = []
for c in ADDONS:
    total = tabelas[c]["iv"].sum()
    sem_internet = tabelas[c].loc["No internet service", "iv"]
    linhas.append({"variavel": c, "iv": total, "parcela_sem_internet": sem_internet, "restante": total - sem_internet})
pd.DataFrame(linhas).sort_values("restante", ascending=False, ignore_index=True).round(4)

,variavel,iv,parcela_sem_internet,restante
0,OnlineSecurity,0.7233,0.3362,0.3871
1,TechSupport,0.7132,0.3362,0.3770
2,OnlineBackup,0.5319,0.3362,0.1956
3,DeviceProtection,0.5016,0.3362,0.1654
4,StreamingMovies,0.3999,0.3362,0.0637
5,StreamingTV,0.3954,0.3362,0.0592


**Resposta:** sim. Os seis serviços e `InternetService` dividem a mesma parcela de 0,336, que vem dos 1.233 clientes sem internet (7,2% de churn). Sem ela, sobram cerca de 0,39 em `OnlineSecurity`, 0,38 em `TechSupport`, 0,20 em `OnlineBackup`, 0,17 em `DeviceProtection` e só 0,06 em cada streaming. **Streaming quase só repete a informação de ter ou não internet.**

O restante é uma aproximação: mostra quanto do IV total vem dos clientes com internet, sem recalcular as proporções só nesse grupo.

## Limites desta análise

- **O IV não enxerga redundância:** cada variável é avaliada sozinha, então os valores não se somam. `tenure` e `TotalCharges` carregam em boa parte a mesma informação, assim como `MonthlyCharges` e os serviços.
- **`tenure` depende de uma resposta da TS:** se ele para de contar no cancelamento, parte do IV alto pode vir disso, e não de um sinal disponível antes de o cliente sair.
- **O IV não enxerga interações:** esse estudo fica para o notebook 05.